# Mini-SGLang：真实单步推理 Trace

这份 notebook 会在真实 mini-sglang 函数入口调用 `debugpy.breakpoint()`，不是手动状态演示。

在 JupyterLab 中：

1. 打开左侧 **Debugger**；
2. 运行当前关的 code cell；
3. 执行暂停时，在 **Call Stack** 选择 `minisgl/...` 帧；
4. 用 **Step Into / Step Over / Continue** 走真实源码；
5. 函数返回后再回答当前问题。答对才解锁下一关。

第一次运行 Setup 会加载模型；之后不要重跑 Setup。需要重新开始时，执行 **Kernel → Restart Kernel**，再从 Setup 开始。

In [2]:
# Setup: this cell is safe to rerun in the same kernel.
import os
import sys
from pathlib import Path
from functools import wraps

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("MINISGL_DISABLE_OVERLAP_SCHEDULING", "1")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ["PATH"] = "/root/miniconda3/bin:" + os.environ.get("PATH", "")

ROOT = Path("/root/workspace/mini-sglang")
if str(ROOT / "python") not in sys.path:
    sys.path.insert(0, str(ROOT / "python"))

import torch
import debugpy
from minisgl.core import SamplingParams
from minisgl.llm import LLM
from minisgl.llm.llm import RequestStatus
from minisgl.message import UserMsg
from minisgl.scheduler.scheduler import Scheduler
from minisgl.scheduler.prefill import PrefillManager
from minisgl.engine.engine import Engine
from minisgl.engine.config import EngineConfig

MODEL = "/root/workspace/models/Qwen2.5-0.5B-Instruct"
PROMPT = "Explain KV cache in one short sentence."
MAX_TOKENS = 2

# EngineConfig otherwise uses fixed tcp://127.0.0.1:2333. A notebook kernel
# needs its own rendezvous port, so a second trace does not collide with another kernel.
TRACE_DISTRIBUTED_PORT = 20000 + os.getpid() % 10000
EngineConfig.distributed_addr = property(
    lambda self: f"tcp://127.0.0.1:{TRACE_DISTRIBUTED_PORT}"
)

if "lesson" not in globals():
    lesson = {"unlocked": 0, "wrapped": set()}

if "llm" not in globals():
    # No torch.cuda API may run before this line.
    print("Creating LLM; Engine initializes CUDA once...")
    llm = LLM(
        model_path=MODEL,
        dtype=torch.bfloat16,
        max_running_req=1,
        max_extend_tokens=256,
        page_size=1,
        cache_type="naive",
        cuda_graph_max_bs=0,
    )
else:
    print("Reusing existing LLM; CUDA is not initialized again.")

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0))


def require_unlocked(previous):
    if lesson["unlocked"] != previous:
        raise RuntimeError(f"Locked: complete stage {previous + 1} first.")


def unlock(level, question, accepted):
    require_unlocked(level - 1)
    answer = input(question + "\n> ").strip().lower()
    if not any(word in answer for word in accepted):
        raise RuntimeError("Not correct. This stage remains locked; inspect the output/debugger and retry.")
    lesson["unlocked"] = level
    print(f"Stage {level} passed.")


def debug_wrap(owner, name, label):
    key = (owner, name)
    if key in lesson["wrapped"]:
        return
    original = getattr(owner, name)

    @wraps(original)
    def wrapped(*args, **kwargs):
        print(f"\n{'=' * 70}\nDEBUG STOP: {label}\nUse JupyterLab Debugger → Call Stack → minisgl frame.\n{'=' * 70}", flush=True)
        debugpy.breakpoint()
        return original(*args, **kwargs)

    setattr(owner, name, wrapped)
    lesson["wrapped"].add(key)

print("Setup complete. Run Stage 1.")

Reusing existing LLM; CUDA is not initialized again.
torch: 2.7.0+cu128
cuda: True
gpu: NVIDIA GeForce RTX 3090
Setup complete. Run Stage 1.


## Stage 1 — Tokenization

真实调用 `LLM._tokenize_one`。暂停后可 Step Into 看 tokenizer 调用；Continue 后会打印 token IDs。

问题：prompt 变成了什么？

In [ ]:
require_unlocked(0)
debug_wrap(LLM, "_tokenize_one", "LLM._tokenize_one")
params = SamplingParams(temperature=0.0, max_tokens=MAX_TOKENS)
input_ids = llm._tokenize_one(PROMPT)
llm.status_map = {0: RequestStatus(uid=0, input_ids=input_ids.tolist(), output_ids=[])}
print("token IDs:", input_ids.tolist())
print("device:", input_ids.device)
unlock(1, "Stage 1: prompt became what?", ["token", "ids"])



DEBUG STOP: LLM._tokenize_one
Use JupyterLab Debugger → Call Stack → minisgl frame.


## Stage 2 — Request arrival

真实调用 `Scheduler._process_one_msg`。在 debugger 中进入 `scheduler.py`，看 `UserMsg` 如何被放入 prefill manager。

问题：返回后请求在 pending 还是 decode running？

In [ ]:
require_unlocked(1)
debug_wrap(Scheduler, "_process_one_msg", "Scheduler._process_one_msg")
llm._process_one_msg(UserMsg(uid=0, input_ids=input_ids, sampling_params=params))
print("pending:", [p.uid for p in llm.prefill_manager.pending_list])
print("decode running:", [r.uid for r in llm.decode_manager.running_reqs])
unlock(2, "Stage 2: answer pending or decode running?", ["pending"])


## Stage 3 — Prefill scheduling

真实调用 `PrefillManager.schedule_next_batch`。Step Into 后看 `PrefillAdder` 创建 `Req`。

问题：为什么这是 prefill 而不是 decode？

In [ ]:
require_unlocked(2)
debug_wrap(PrefillManager, "schedule_next_batch", "PrefillManager.schedule_next_batch")
batch = llm.prefill_manager.schedule_next_batch(llm.prefill_budget)
print("phase:", batch.phase)
print("request state:", [(r.uid, r.cached_len, r.device_len, r.remain_len) for r in batch.reqs])
unlock(3, "Stage 3: why prefill? Answer prompt/input.", ["prompt", "input"])


## Stage 4 — GPU metadata

真实调用 `Scheduler._prepare_batch`。Step Into 后看 page allocation、positions、mapping 和 attention metadata。

问题：这些 metadata 帮 GPU 找什么？

In [ ]:
require_unlocked(3)
debug_wrap(Scheduler, "_prepare_batch", "Scheduler._prepare_batch")
forward_input = llm._prepare_batch(batch)
print("positions:", tuple(batch.positions.shape))
print("input mapping:", [tuple(x.shape) for x in forward_input.input_tuple])
print("write mapping:", [tuple(x.shape) for x in forward_input.write_tuple])
print("out_loc:", tuple(batch.out_loc.shape))
unlock(4, "Stage 4: answer addresses/KV cache/metadata.", ["address", "kv", "cache", "metadata"])


## Stage 5 — Model forward and sampling

真实调用 `Scheduler._forward`，然后进入真实 `Engine.forward_batch`。在 Engine 中可看到：`model.forward()` 先执行，之后才 `sampler.sample()`。

问题：forward 和 sampling 谁在前？

In [ ]:
require_unlocked(4)
debug_wrap(Scheduler, "_forward", "Scheduler._forward")
debug_wrap(Engine, "forward_batch", "Engine.forward_batch")
with llm.engine_stream_ctx:
    forward_output = llm._forward(forward_input)
print("next token GPU:", forward_output.next_tokens_gpu)
print("next token CPU:", forward_output.next_tokens_cpu)
unlock(5, "Stage 5: answer forward before sampling.", ["forward", "before"])


## Stage 6 — Result handling and decode

真实调用 `_process_last_data`，再真实执行一轮 decode。Step Into 可以看 token 如何 append、何时加入 decode、何时释放资源。

问题：后续 token 为什么进入 decode？完成时需要什么操作？

In [ ]:
require_unlocked(5)
debug_wrap(Scheduler, "_process_last_data", "Scheduler._process_last_data")
llm._process_last_data((forward_input, forward_output))
print("after prefill:", [(r.uid, r.remain_len) for r in llm.decode_manager.running_reqs])

if llm.decode_manager.runnable:
    decode_batch = llm.decode_manager.schedule_next_batch()
    decode_input = llm._prepare_batch(decode_batch)
    with llm.engine_stream_ctx:
        decode_output = llm._forward(decode_input)
    llm._process_last_data((decode_input, decode_output))

print("token IDs:", llm.status_map[0].output_ids)
print("text:", llm.tokenizer.decode(llm.status_map[0].output_ids))
unlock(6, "Stage 6: answer decode and release/free.", ["decode"])


## Done

真实调用栈已经经过：

```text
LLM._tokenize_one
→ Scheduler._process_one_msg
→ PrefillManager.schedule_next_batch
→ Scheduler._prepare_batch
→ Scheduler._forward
→ Engine.forward_batch
→ Scheduler._process_last_data
→ decode
```


In [ ]:
require_unlocked(6)
llm.shutdown()
del llm
print("resources released; restart kernel before another trace")